In [1]:
%load_ext autoreload
%autoreload 2

# Part 3: Collaborative Filtering using K-Nearest Neighbors
**Collaborative Filtering** is based on idea that "similar users like similar items". This algorithm using **K-Nearest Neighbors** algorithm and can be approached in two ways:
* **User-User CF:** Find users with similar rating histories to predict preferences of a user
* **Item-Item CF:** Find items that are rated similarly by same groups of users to recommed items related to what a user has already liked

## Import libraries

In [2]:
import pandas as pd 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse 

## 1.Import files

### Train-Test data (Utility Matrix)

In [4]:
ratings_train=pd.read_csv('../data/processed/ratings_b_train.csv',header=0).values
ratings_test=pd.read_csv('../data/processed/ratings_b_test.csv',header=0).values

In [5]:
ratings_train

array([[        0,         0,         5, 874965758],
       [        0,         1,         3, 876893171],
       [        0,         2,         4, 878542960],
       ...,
       [      942,      1187,         3, 888640250],
       [      942,      1227,         3, 888640275],
       [      942,      1329,         3, 888692465]], shape=(90570, 4))

## 2.Build Collaborative Filtering Recommend System

### 2.1.Similarity functions
To find "neighbors", we use **Cosine Similarity**. Unlike Euclide distance, it measures **angle** between two vectors:
$$similarity(\mathbf{u_1},\mathbf{u_2})=\frac{\mathbf{u_1} \cdot \mathbf{u_2}}{||\mathbf{u_1}|| \cdot ||\mathbf{u_2}||}$$

### 2.2.User-User Collaborative Filtering

#### 2.2.1.Normalize data
We subtract each cell of utility matrix (not sparse) by averange of values by columns and replace missing value by 0. The reasons why we have to have the step because:
* Subtracting the averange of each columns result in both negative and positive values within each column. The positive values represent that a user likes a movie, nagative values represent that a user do not like a movie. Zero values represent that  an undetermined state of whether the user likes the item.
* An advantage of storing utility matrix under a sparse matrix that the dimensions of utility matrix are very large because of the huge number of users and items. Observed that the number of known data is very small compared to the size of utility matrix, so that is better to store the data under sparse matrix, ie just save non-zero values and those positive.

#### 2.2.2.Rating prediction

Popular function to predict rating of user $u$ for movie $i$:
$$\hat{y}_{i,u}=\frac{\sum_{u_j \in \mathcal{N}(u,i)} \bar{y}_{i,u_j} \text{sim}(u,u_j)}{\sum_{u_j \in \mathcal{N}(u,i)} |\text{sim}(u,u_j)|}$$
**Where**
* $\mathcal{N}(u,i)$: The set of k users in neighborhood (ie have highest **similarity**) of users $u$ who have rated movie $i$
* $\text{sim}(u,u_j)$: The **similarity score** between user $u$ and $u_j$ 
* $\bar{y}_{i,u_j}$: The rating of neighbor $u_j$ for movie $i$ (already mean-centered)

To translate normalized predictions back into a human-readable format, we must perform Denormalization. This is achieved by adding the user-specific mean ratings back to each corresponding column of the predicted matrix $\hat{Y}$.

### 2.3.Item-Item Collaborative Filtering 

#### 2.3.1.Analysis User-User Collaborative Filtering

* In fact, the number of users is very large compared to the number of items, so that the size of user-user similary matrix is absolutely big. This reason makes storing user-user matrix in some situations is very challenging.
* The utility matrix is usually sparse. Because the number of users is very large compared to the number of items, many columns of utility matrix are very sparse (ie just a few of elements is non-zero), the reason can be that the users are lazy to rate movies. So that as soon as u user change ratings or add new ratings, the mean of ratings and the normalized rating vector change significant, result in the computing Similarity matrix-actually waste much memory and time, the similarity matrix also recompute.

#### 2.3.2.Analysis Item-Item Collaborative Filtering

Conversely, calculating similarity between items and recommending those similar to a user's favorites offers the following key benefits:
* Efficiency in Storage and Computation: Since the number of items is much smaller than the number of users, the Similarity Matrix in this case is significantly smaller. This makes it far more efficient to store and speeds up subsequent computational steps.
* Model Stability: While the number of known entries in the Utility Matrix remains the same, there are fewer rows (items) than columns (users). Consequently, on average, each row contains more known ratings than each column. This is intuitive since a single item is often rated by many users. As a result, the mean value of each row is less sensitive to a few new ratings, meaning the Similarity Matrix stays stable and requires less frequent updates.

#### 2.3.3. Computing

Item-Item CF can be received from User-User CF by transposing Utility Matrix, treating items as users. After computing the result, we transpose again to receive the result

### 2.3.Define class for system